# StereoQueerEval 2027 — Task A: Class-Aware Multi-Head Cross-Attention (End-to-End)

Notebook này thiết lập quy trình huấn luyện **End-to-End** cho **Task A (Stereotype Classification - 2 classes: `no`, `yes`)** sử dụng kiến trúc tùy chỉnh nâng cao:

```
[INPUT SEQUENCE]
Comment: <C> ... </C> | Title: <T> ... </T> | Description: <D> ... </D>
                                   │
                                   ▼
┌─────────────────────────────────────────────────────────────────────────────┐
│ BACKBONE & POST-ENCODER CONTEXTUAL ROLE INJECTION                           │
│  Input Token IDs  [B, S] ──────► mmBERT Encoder ────► H_mmBERT  [B, S, 768] │
│  Role IDs         [B, S] ──────► Role Embeddings ───► E_role    [B, S, 768] │
│                     H_final = LayerNorm(H_mmBERT + E_role) ───┴─► [B, S, 768]│
└──────────────────────────────────────┬──────────────────────────────────────┘
                                       ▼
┌─────────────────────────────────────────────────────────────────────────────┐
│ LAYER 1 & 2: CONSECUTIVE CROSS-ATTENTION DECODER STACK (L=3 hops)           │
│  Learned Stereotype Queries: Q_base = [q_NonStereotype, q_Stereotype] [2,768]│
│  A = Softmax(Q·Kᵀ / √d_h)       ∈ [B, 2, S]  (Interpretability Attention Map)│
│  Multi-Scale: H20 (local) -> H21 (local) -> H22 (global) ──► Z ∈ [B, 2, 768] │
└──────────────────────────────────────┬──────────────────────────────────────┘
                                       ▼
┌─────────────────────────────────────────────────────────────────────────────┐
│ LAYER 2: QUERY INTERACTION LAYER (MHSA) [Ablation Hypothesis H2]            │
│  Self-Attention: NonStereotype ↔ Stereotype                                 │
│  Z' = LayerNorm(Z + Dropout(MHSA(Q=Z, K=Z, V=Z))) ──► Tensor Z' ∈ [B, 2, 768]│
└──────────────────────────────────────┬──────────────────────────────────────┘
                                       ▼
┌─────────────────────────────────────────────────────────────────────────────┐
│ LAYER 3: SHARED SCORING HEAD & TASK A BRIDGE                                │
│  z'_c [B, 768] ──► Joint Head f_θ ──► Raw Scores s ∈ [B, 2]                 │
│  Probabilities p = Softmax(s)                                               │
│  Task A Bridge: h_A = ∑_c (p_c · z'_c) ∈ [B, 768]                           │
└─────────────────────────────────────────────────────────────────────────────┘
```


## 1. Clone hoặc Cập nhật Mã Nguồn từ GitHub

In [ ]:
import os
import sys

REPO_URL = "https://github.com/your-username/stereoqueer-pipeline.git"
REPO_DIR = "stereoqueer-pipeline"

if not os.path.exists(REPO_DIR):
    print(f"Cloning {REPO_URL}...")
    !git clone {REPO_URL} {REPO_DIR}
    %cd {REPO_DIR}
else:
    print(f"Repository directory '{REPO_DIR}' already exists.")
    if os.path.isdir(REPO_DIR):
        %cd {REPO_DIR}
        !git pull

if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

print("Current Working Directory:", os.getcwd())
!ls -la

## 2. Cài đặt Thư viện & Smoke Test Pipeline

In [ ]:
!pip install -q torch transformers datasets accelerate scikit-learn pandas numpy matplotlib seaborn

import torch
print("PyTorch Version:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU Device:", torch.cuda.get_device_name(0))
    print("Memory Allocated:", round(torch.cuda.memory_allocated(0)/1024**3, 2), "GB")

# Chạy smoke test toàn bộ pipeline
!python smoke_test.py

## 3. Chuẩn bị Dữ liệu Huấn luyện

In [ ]:
import os
import glob

os.makedirs("data", exist_ok=True)
os.makedirs("checkpoints_task_a", exist_ok=True)

kaggle_paths = [
    "/kaggle/input/datasets/amnpham/lgbtqia/LGBT",
    "/kaggle/input/lgbtqia/LGBT",
    "/kaggle/input/stereoqueereval",
    "/kaggle/input"
]

tsv_files = []
for kp in kaggle_paths:
    if os.path.exists(kp):
        found = glob.glob(f"{kp}/**/*.tsv", recursive=True) + glob.glob(f"{kp}/*.tsv")
        if found:
            tsv_files.extend(found)
            break

if not tsv_files:
    tsv_files = [os.path.join("data", f) for f in os.listdir("data") if f.endswith(".tsv")]

if not tsv_files:
    print("Chưa phát hiện file TSV. Đang tạo sample dataset để kiểm thử pipeline...")
    !python generate_sample_data.py
    tsv_files = [os.path.join("data", f) for f in os.listdir("data") if f.endswith(".tsv")]

tsv_files = sorted(list(set(tsv_files)))
print(f"Đã tìm thấy {len(tsv_files)} file TSV:")
for f in tsv_files:
    print(" -", f)

## 4. Load Data & Stratified Multi-Lingual Group Splitting (`StratifiedGroupKFold`)

Phân chia dữ liệu theo **cụm video (`yt_title`)** đồng thời phân tầng cân bằng tỷ lệ **ngôn ngữ (`lang`)** và **nhãn định kiến (`st_y`)** giữa Train (80%) và Validation (20%), đảm bảo tuyệt đối không rò rỉ ngữ cảnh video.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.utils.class_weight import compute_class_weight
from pipeline.config import PipelineConfig, STEREOTYPE_CLASSES, STEREOTYPE2IDX, IDX2STEREOTYPE
from pipeline.data import DataPipeline
from pipeline.task_a_data import TaskARoleDataset
from pipeline.utils import set_seed, get_seeded_generator, seed_worker
from pipeline.models.task_a_class_aware import (
    TaskAClassAwareAttentionModel,
    ROLE_PAD, ROLE_TITLE, ROLE_DESC, ROLE_COMMENT
)

config = PipelineConfig(
    task="stereoqueer",
    target_task="st", # Task A (Stereotype)
    model_type="task_a_class_aware",
    mmbert_model_name="jhu-clsp/mmbert-base",
    batch_size=32,
    max_length=256,
    num_decoder_layers=3,
    use_multiscale_layers=True,
    decoder_ffn_dim=1536,
    dropout=0.25,
    weight_decay=0.01,
    learning_rate=2.5e-4,
    loss_type="focal",
    focal_gamma=2.0,
    label_smoothing=0.05,
    two_phase=True,
    freeze_phase_epochs=3,
    unfreeze_phase_epochs=15,
    unfreeze_layers=3,
    unfreeze_lr=1.5e-5,
    head_unfreeze_lr=6e-5,
    patience=5,
    output_dir="checkpoints_task_a"
)

data_pipeline = DataPipeline(config)
df_all = data_pipeline.load_data(tsv_files)

# Khóa phân tầng = ngôn ngữ + nhãn (vd: 'IT_1.0'), để cân bằng cả hai cùng lúc
strat_key = (df_all['lang'].astype(str) + '_' + df_all['st_y'].astype(str)).values
groups = df_all['yt_title'].values

# Tỉ lệ chuẩn của từng ô (lang, label) trên toàn bộ dữ liệu
all_ratio = df_all.assign(k=strat_key).groupby('k').size() / len(df_all)

def split_score(val_idx):
    """Độ lệch tối đa giữa tỉ lệ ô (lang,label) trong val so với toàn bộ + lệch kích thước so với 20%."""
    v = pd.Series(strat_key[val_idx]).value_counts() / len(val_idx)
    v = v.reindex(all_ratio.index).fillna(0)
    ratio_dev = (v - all_ratio).abs().max()
    size_dev = abs(len(val_idx) / len(df_all) - 0.20)
    return ratio_dev + size_dev

best = None
for seed in range(1000):
    sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=seed)
    tr_idx, va_idx = next(iter(sgkf.split(df_all, strat_key, groups=groups)))
    s = split_score(va_idx)
    if best is None or s < best[0]:
        best = (s, seed, tr_idx, va_idx)

score, best_seed, train_idx, val_idx = best
df_train = df_all.iloc[train_idx].reset_index(drop=True)
df_val = df_all.iloc[val_idx].reset_index(drop=True)

data_pipeline.df_train = df_train
data_pipeline.df_val = df_val
config.random_seed = best_seed
set_seed(best_seed)

# --- Kiểm tra phân phối ---
overlap = len(set(df_train['yt_title']) & set(df_val['yt_title']))
print(f"Seed chọn tối ưu: {best_seed} | Score lệch: {score:.4f}")
print(f"Train: {len(df_train)} ({len(df_train)/len(df_all):.1%}) | "
      f"Val: {len(df_val)} ({len(df_val)/len(df_all):.1%}) | Video trùng lặp: {overlap}")

print("\nTỉ lệ nhãn dương (st_y=1) theo ngôn ngữ:")
cmp = pd.DataFrame({
    'all_n':     df_all.groupby('lang').size(),
    'train_n':   df_train.groupby('lang').size(),
    'val_n':     df_val.groupby('lang').size(),
    'all_pos':   df_all.groupby('lang')['st_y'].mean().round(3),
    'train_pos': df_train.groupby('lang')['st_y'].mean().round(3),
    'val_pos':   df_val.groupby('lang')['st_y'].mean().round(3),
})
print(cmp)

print("\nTỉ lệ ngôn ngữ trong từng tập:")
print(pd.DataFrame({
    'all':   df_all['lang'].value_counts(normalize=True),
    'train': df_train['lang'].value_counts(normalize=True),
    'val':   df_val['lang'].value_counts(normalize=True),
}).round(3))

# Tính toán trọng số lớp cân bằng
train_st_labels = (df_train['stereotype'] == 'yes').astype(int).values
weights = compute_class_weight('balanced', classes=np.array([0, 1]), y=train_st_labels)
config.class_weights = weights.tolist()
print("\nBalanced Class Weights [no, yes]:", config.class_weights)

## 5. Tokenizer & Explicit Role Injection Dataset (`TaskARoleDataset`)

In [ ]:
from transformers import AutoTokenizer
from torch.utils.data import DataLoader

print(f"Loading ModernBERT/mmBERT tokenizer: {config.mmbert_model_name}...")
tokenizer = AutoTokenizer.from_pretrained(config.mmbert_model_name)

train_ds = TaskARoleDataset(df_train, tokenizer, max_len=config.max_length)
val_ds = TaskARoleDataset(df_val, tokenizer, max_len=config.max_length)

train_loader = DataLoader(
    train_ds,
    batch_size=config.batch_size,
    shuffle=True,
    generator=get_seeded_generator(config.random_seed),
    worker_init_fn=seed_worker,
    pin_memory=torch.cuda.is_available(),
    num_workers=2
)
val_loader = DataLoader(
    val_ds,
    batch_size=config.batch_size,
    shuffle=False,
    pin_memory=torch.cuda.is_available(),
    num_workers=2
)

# Kiểm tra 1 mẫu đầu vào và role IDs
sample_input_ids, sample_mask, sample_roles, sample_idx, sample_st, sample_hs, sample_tg = train_ds[0]
print("Sample Input IDs shape:", sample_input_ids.shape)
print("Sample Role IDs shape:", sample_roles.shape)
print("Sample Label ST (0=no, 1=yes):", sample_st.item(), f"({IDX2STEREOTYPE[sample_st.item()]})")


## 6. Khởi tạo Kiến trúc Task A: Class-Aware Cross-Attention

In [ ]:
from transformers import AutoModel

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

print(f"Loading backbone: {config.mmbert_model_name}...")
backbone = AutoModel.from_pretrained(config.mmbert_model_name)

model = TaskAClassAwareAttentionModel(
    mmbert_model=backbone,
    d_model=config.mmbert_dim,
    num_heads=config.num_heads,
    dropout=config.dropout,
    use_query_interaction=True,
    num_queries=2
).to(device)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total Parameters: {total_params:,} | Trainable: {trainable_params:,}")

dummy_ids = sample_input_ids.unsqueeze(0).to(device)
dummy_mask = sample_mask.unsqueeze(0).to(device)
dummy_roles = sample_roles.unsqueeze(0).to(device)

with torch.no_grad():
    logits, h_A, attn_map = model(dummy_ids, dummy_mask, dummy_roles, return_attention_map=True)

print("Forward Output Logits shape:", logits.shape)       # [1, 2]
print("Task A Bridge h_A shape:", h_A.shape)             # [1, 768]
print("Attention Map shape:", attn_map.shape)             # [1, 2, 256]
print("Probabilities:", torch.softmax(logits, dim=-1).cpu().numpy())

## 7. Huấn Luyện 2-Phase End-to-End

In [ ]:
from pipeline.task_a_trainer import TaskATrainer

trainer = TaskATrainer(
    model=model,
    config=config,
    train_loader=train_loader,
    val_loader=val_loader,
    df_val=df_val
)

print("Bắt đầu huấn luyện mô hình Task A...")
results = trainer.train()

print("\nHuấn luyện hoàn tất!")
print("Best Checkpoint File:", results['checkpoint_path'])
print("Final Metrics:", results['final_metrics'])

## 8. Trực Quan Hóa Quá Trình Huấn Luyện (Loss & Macro-F1 Curves)

In [ ]:
import matplotlib.pyplot as plt

history = results['history']
if history:
    epochs = [h['epoch'] for h in history]
    train_loss = [h['train_loss'] for h in history]
    val_loss = [h['val_loss'] for h in history]
    macro_f1 = [h['st_macro_f1'] for h in history]

    plt.figure(figsize=(12, 4))
    
    plt.subplot(1, 2, 1)
    plt.plot(epochs, train_loss, label='Train Loss', marker='o')
    plt.plot(epochs, val_loss, label='Val Loss', marker='s')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.title('Training & Validation Loss')
    plt.grid(True, alpha=0.3)
    plt.legend()

    plt.subplot(1, 2, 2)
    plt.plot(epochs, macro_f1, label='Val Macro-F1', color='green', marker='^')
    plt.xlabel('Epoch')
    plt.ylabel('Macro-F1')
    plt.title('Task A Stereotype Macro-F1 Over Epochs')
    plt.grid(True, alpha=0.3)
    plt.legend()

    plt.tight_layout()
    plt.show()
else:
    print("No history recorded.")

## 9. Đánh Giá Chi Tiết & Ma Trận Nhầm Lẫn (Confusion Matrix)

In [ ]:
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix

pred_csv_path = os.path.join(config.output_dir, "task_a_val_predictions.csv")
if os.path.exists(pred_csv_path):
    df_preds = pd.read_csv(pred_csv_path)
    y_true = df_preds['stereotype'].tolist()
    y_pred = df_preds['pred_stereotype'].tolist()
    
    labels = ['no', 'yes']
    print("=== CLASSIFICATION REPORT (Task A - Stereotype) ===")
    print(classification_report(y_true, y_pred, target_names=labels, zero_division=0))
    
    cm = confusion_matrix(y_true, y_pred, labels=labels)
    plt.figure(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Purples', xticklabels=labels, yticklabels=labels)
    plt.xlabel('Predicted')
    plt.ylabel('True Gold Label')
    plt.title('Confusion Matrix - Task A Class-Aware MHCA')
    plt.show()
else:
    print(f"File {pred_csv_path} not found.")

## 10. Interpretability: Trực Quan Hóa Class-Conditioned Attention Map

In [ ]:
model.eval()
test_idx = 0
for i in range(len(val_ds)):
    if val_ds[i][4].item() == 1: # Tìm mẫu có stereotype ('yes')
        test_idx = i
        break

inp_ids, att_mask, r_ids, sample_idx, true_st, _, _ = val_ds[test_idx]
with torch.no_grad():
    out_s, out_hA, attn_weights = model(
        inp_ids.unsqueeze(0).to(device),
        att_mask.unsqueeze(0).to(device),
        r_ids.unsqueeze(0).to(device),
        return_attention_map=True
    )

tokens = tokenizer.convert_ids_to_tokens(inp_ids.tolist())
valid_len = int(att_mask.sum().item())
tokens = tokens[:valid_len]
attn_map_np = attn_weights[0, :, :valid_len].cpu().numpy() # [2, valid_len]

plt.figure(figsize=(14, 3))
sns.heatmap(
    attn_map_np,
    cmap="PuRd",
    xticklabels=[t if len(t) < 12 else t[:10] + '..' for t in tokens],
    yticklabels=['NonStereotype Query', 'Stereotype Query']
)
plt.xticks(rotation=90, fontsize=8)
plt.title(f'Attention Distribution Across Tokens (True Label: {IDX2STEREOTYPE[true_st.item()]})')
plt.show()

## 11. Task A Representation Bridge ($h_A$ Extraction)

Xuất biểu diễn $h_A = \sum_c (p_c \cdot z'_c) \in [B, 768]$ để làm đặc trưng bổ trợ cho Task B hoặc Task C.

In [ ]:
print("Trích xuất Task A Bridge representations h_A cho tập Validation...")
model.eval()
h_A_list = []

with torch.no_grad():
    for batch in val_loader:
        i_ids, a_mask, r_ids, sample_idx, _, _, _ = batch
        _, h_A, _ = model(i_ids.to(device), a_mask.to(device), r_ids.to(device))
        h_A_list.append(h_A.cpu().numpy())

h_A_all = np.concatenate(h_A_list, axis=0)
print(f"Extracted h_A matrix shape: {h_A_all.shape} ([N_val, 768])")

bridge_save_path = os.path.join(config.output_dir, "val_task_a_bridge_h_A.npy")
np.save(bridge_save_path, h_A_all)
print(f"Saved bridge vectors to: {bridge_save_path}")
print("Hoàn tất toàn bộ quy trình Task A!")